# Prepare Canonical Filtered Inputs

Creates the shared filtered h5ad + cell/gene allowlists that ALL downstream pipelines use.

**Run this FIRST, once per dataset.**

Change `DATASET` below to `"pbmc3k"` or `"lung65k"` and re-run.

In [1]:
# === CHANGE THIS TO SWITCH DATASETS ===
DATASET = "lung65k"   # or "lung65k"

import os
CONFIG_PATH = "benchmark_config.json"
assert os.path.exists(CONFIG_PATH), f"Config not found: {CONFIG_PATH}"

In [2]:
%%time
import json
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

print(f"Scanpy version: {sc.__version__}")
print(f"Dataset: {DATASET}")

Scanpy version: 1.12
Dataset: lung65k
CPU times: user 3.3 s, sys: 104 ms, total: 3.41 s
Wall time: 1.19 s


<timed exec>:8: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead


In [3]:
# Load config
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)
gcfg = config["global"]
dcfg = config["datasets"][DATASET]

print("Dataset:", dcfg["name"])
print("Input:", dcfg["input_path"])
print("Canonical h5ad:", dcfg["canonical_h5ad"])
print("Filters: min_cells=%d, min_genes=%d, max_genes=%d" % (
    dcfg["min_cells"], dcfg["min_genes"], dcfg["max_genes"]))
print("max_pct_mt:", dcfg["max_pct_mt"])

Dataset: Human Lung Cell 65K
Input: /home/seyun/thesis_validation/lung_65k/data/krasnow_hlca_10x.sparse.h5ad
Canonical h5ad: write/lung_65k_canonical_filtered.h5ad
Filters: min_cells=3, min_genes=200, max_genes=6000
max_pct_mt: None


In [4]:
%%time
# Load raw data
if dcfg["input_type"] == "10x_mtx":
    adata = sc.read_10x_mtx(dcfg["input_path"], var_names="gene_symbols", cache=False)
elif dcfg["input_type"] == "h5ad":
    adata = sc.read_h5ad(dcfg["input_path"])
else:
    raise ValueError(f"Unsupported input_type: {dcfg['input_type']}")

adata.var_names = pd.Index(adata.var_names.astype(str))
adata.var_names_make_unique()

# Preserve raw counts
if sparse.issparse(adata.X):
    adata.X = adata.X.tocsr()
    adata.layers["counts"] = adata.X.copy()
else:
    adata.X = np.asarray(adata.X)
    adata.layers["counts"] = adata.X.copy()

print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

Loaded: 65,662 cells × 26,485 genes
CPU times: user 145 ms, sys: 219 ms, total: 364 ms
Wall time: 364 ms


In [5]:
# Annotate QC variables
RIBO_RE = re.compile(r"^RP[SL]", flags=re.IGNORECASE)
HB_RE = re.compile(r"^HB(?!P)", flags=re.IGNORECASE)

var_upper = pd.Index(adata.var_names.astype(str)).str.upper()
adata.var["mt"] = var_upper.str.startswith("MT-")
adata.var["ribo"] = [bool(RIBO_RE.match(g)) for g in var_upper]
adata.var["hb"] = [bool(HB_RE.match(g)) for g in var_upper]

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    inplace=True,
    log1p=False,
    percent_top=None,
)
print("QC metrics computed")

QC metrics computed


In [6]:
%%time
# Filter genes
before = (adata.n_obs, adata.n_vars)
sc.pp.filter_genes(adata, min_cells=dcfg["min_cells"])
print(f"Gene filter: {before[1]:,} → {adata.n_vars:,} genes")

# Filter cells
cell_mask = (
    (adata.obs["n_genes_by_counts"] >= dcfg["min_genes"]) &
    (adata.obs["n_genes_by_counts"] <= dcfg["max_genes"])
)
if dcfg["max_pct_mt"] is not None and "pct_counts_mt" in adata.obs.columns:
    cell_mask &= adata.obs["pct_counts_mt"] < dcfg["max_pct_mt"]
    print(f"mt% filter applied: < {dcfg['max_pct_mt']}")
else:
    print("mt% filter: NOT APPLIED")

adata = adata[cell_mask].copy()
print(f"\nCanonical filtered: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

Gene filter: 26,485 → 20,901 genes
mt% filter: NOT APPLIED

Canonical filtered: 65,462 cells × 20,901 genes
CPU times: user 1.24 s, sys: 621 ms, total: 1.86 s
Wall time: 1.86 s


In [7]:
%%time
# Save canonical outputs
os.makedirs(os.path.dirname(dcfg["canonical_h5ad"]), exist_ok=True)

adata.write_h5ad(dcfg["canonical_h5ad"], compression="gzip")

with open(dcfg["canonical_cells_txt"], "w") as f:
    for bc in adata.obs_names.astype(str):
        f.write(f"{bc}\n")

with open(dcfg["canonical_genes_txt"], "w") as f:
    for gene in adata.var_names.astype(str):
        f.write(f"{gene}\n")

summary = {
    "dataset": dcfg["name"],
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "filters": {
        "min_cells": dcfg["min_cells"],
        "min_genes": dcfg["min_genes"],
        "max_genes": dcfg["max_genes"],
        "max_pct_mt": dcfg["max_pct_mt"],
    },
    "global_defaults": gcfg,
}
with open(dcfg["canonical_summary_json"], "w") as f:
    json.dump(summary, f, indent=2)

print("Wrote:")
print(f"  {dcfg['canonical_h5ad']}")
print(f"  {dcfg['canonical_cells_txt']}")
print(f"  {dcfg['canonical_genes_txt']}")
print(f"  {dcfg['canonical_summary_json']}")

Wrote:
  write/lung_65k_canonical_filtered.h5ad
  write/lung_65k_canonical_cells.txt
  write/lung_65k_canonical_genes.txt
  write/lung_65k_canonical_summary.json
CPU times: user 25.5 s, sys: 228 ms, total: 25.7 s
Wall time: 25.8 s
